# Consommer vs exposer le MCP — sur l'instance reelle

Deux sens traversent le fil MCP. Une application **expose** un catalogue d'outils que des agents peuvent appeler ; elle **consomme** aussi, en tant que client, les outils d'autres serveurs. Ce carnet etudie les deux sens **sur l'instance jetable Maison Valmont**, avec des serveurs reels dans la boucle et un modele reel au volant.

Il remplace une version anterieure ou tout etait en memoire, puis une deuxieme ou le serveur consomme etait un mini-serveur `FastMCP` ecrit par le carnet lui-meme — un systeme reel face a un systeme construit par la mesure, ce qui affaiblissait la these. Trois choses changent ici :

1. **Le catalogue expose est le vrai** — les fonctions que le plugin AI Engine expose effectivement, lues par `mwai/v1/mcp/functions`, et le meme serveur est consomme **par le protocole MCP lui-meme** : session JSON-RPC sur `mcp/v1/http`, handshake, `tools/list`, `tools/call`.
2. **Le second serveur est reel et externe** — le serveur MCP `filesystem` officiel (`@modelcontextprotocol/server-filesystem`), joint par stdio, qui tient un rayon de fiches de lecture. Aucun des deux serveurs de la batterie n'est ecrit par ce carnet.
3. **Le chevauchement se mesure sur un agent, avec des repetitions** — un modele reel recoit les deux catalogues fusionnes et une batterie de seize taches, **repetees** pour donner a chaque taux un intervalle de confiance ; la meme batterie passe sur **deux modeles de tailles differentes**, pour separer l'effet modele de l'effet protocole. Jaccard reste la **premiere** approche, pas l'aboutissement.

**Prerequis.** L'instance `instance-jetable` doit etre montee (README de la serie, cinq etapes) et son `.env` renseigne ; un moteur local compatible OpenAI doit repondre, avec `mcp`, `openai`, `requests` et `python-dotenv` ; le second moteur et `npx` (Node.js) pour le serveur filesystem. Voir « Provenance et limites » en fin de carnet.


### Lecture

Trois lignes de configuration, trois choix de protocole qui pesent sur toute la suite. Les **moteurs** sont deux vLLM locaux de la meme famille : un **qwen35-0.8b** en principal (`http://127.0.0.1:8001/v1`) et un **qwen35-4b** en contraste (`http://127.0.0.1:8002/v1`) — aucune cle de provider ne quitte le fichier `.env` de l'instance, et le carnet reste executable sans credit. L'**instance WordPress** est un conteneur jetable sur `http://localhost:8093` (la Maison Valmont), reconstruisable par `docker compose` : les ecritures qu'on va provoquer tombent dans un bac a sable, pas chez un client. Enfin, le choix d'un moteur principal de **0,8 milliard** de parametres rend la mesure volontairement conservatrice — si un tout petit modele distingue deja (ou non) les deux catalogues, un modele de front fera au moins aussi bien — et le moteur de contraste dit en section 5 ce que cette taille ajoute. C'est le sens de la mesure qui suit, pas une course au score.


## 2. Le cote expose : ce que le site publie vraiment

AI Engine 3.7.0 expose ses outils sur une route authentifiee. On lit le catalogue tel qu'il est, sans le reconstituer.


In [2]:
statut, charge = api("/mwai/v1/mcp/functions")
print("statut HTTP :", statut)
outils_exposes = charge["functions"]
print("outils exposes :", len(outils_exposes))
print()
for outil in outils_exposes:
    description = (outil.get("description") or "").replace("\n", " ")[:58]
    print(f"  {outil['name']:30s} [{outil.get('accessLevel', '?'):5s}] {description}")


statut HTTP : 200
outils exposes : 42

  wp_list_plugins                [read ] List installed plugins (returns array of {Name, Version}).
  wp_get_users                   [admin] Retrieve users (fields: ID, user_login, display_name, role
  wp_create_user                 [admin] Create a user. Requires user_login and user_email. Optiona
  wp_update_user                 [admin] Update a user – pass ID plus a “fields” object (user_email
  wp_get_comments                [read ] Retrieve comments (fields: comment_ID, comment_post_ID, co
  wp_create_comment              [write] Insert a comment. Requires post_id and comment_content. Op
  wp_update_comment              [write] Update a comment – pass comment_ID plus fields (comment_co
  wp_delete_comment              [admin] Delete a comment. `force` true bypasses trash.
  wp_get_option                  [admin] Get a single WordPress option value (scalar or array) by k
  wp_update_option               [admin] Create or update a WordPress opt

### Lecture

La reponse HTTP **200** et les **42 outils** affiches meritent une lecture attentive, colonne par colonne. Le nom d'abord : tout le catalogue est un CMS — **posts** (`wp_get_post`, `wp_create_post`, `wp_update_post`, `wp_delete_post`, le snapshot en un appel `wp_get_post_snapshot`, la recherche-remplacement `wp_alter_post`, les blocs Gutenberg `wp_write_blocks`), **commentaires** (cycle complet jusqu'au `wp_delete_comment`), **medias** (`wp_upload_media`, `wp_upload_request`), **taxonomies et termes** (`wp_get_terms`, `wp_add_post_terms`...), **options** (`wp_get_option`, `wp_update_option`), **utilisateurs** (`wp_create_user`, `wp_update_user`). Puis le crochet d'acces : `read` pour consulter, `write` pour ecrire, `admin` pour detruire ou toucher au noyau — la colonne dit quel niveau il FAUDRAIT exiger, pas ce que l'agent respectera. Enfin les deux outsiders `mwai_vision` et `mwai_image`, greffes du plugin AI Engine sur le CMS. Aucun de ces 42 outils ne parle de livre, d'auteur ou de catalogue : le domaine de l'editeur est absent du cote expose.

In [3]:
# Ce que le catalogue expose dit du site : des verbes generiques, aucun outil metier.
familles = {}
for outil in outils_exposes:
    prefixe = outil["name"].split("_", 1)[0]
    familles[prefixe] = familles.get(prefixe, 0) + 1
print("familles de prefixe :", dict(sorted(familles.items())))
metier = [o["name"] for o in outils_exposes if not o["name"].startswith(("wp_", "mwai_"))]
print("outils metier (hors wp_/mwai_) :", metier or "aucun")

# Et les niveaux d'acces, qui disent ce qu'un agent peut oser.
niveaux = {}
for outil in outils_exposes:
    niveaux[outil.get("accessLevel", "?")] = niveaux.get(outil.get("accessLevel", "?"), 0) + 1
print("niveaux d'acces :", dict(sorted(niveaux.items())))


familles de prefixe : {'mwai': 2, 'wp': 40}
outils metier (hors wp_/mwai_) : aucun
niveaux d'acces : {'admin': 10, 'read': 16, 'write': 16}


### Lecture

Le rayon est un dossier reel — trois fiches de lecture vues par le premier `list_directory` — et le handshake stdio est lui aussi reel et complet : `initialize` (serveur **secure-filesystem-server**, **protocole 2025-06-18**, la meme version que cote site), puis `tools/list` qui rend les **12 outils consommes** — `read_file`, `read_multiple_files`, `write_file`, `edit_file`, `create_directory`, `list_directory`, `directory_tree`, `move_file`, `search_files`, `get_file_info`... — chacun avec sa description et son schema d'arguments. La comparaison avec le catalogue expose de la section 2 est frappante, mais elle n'est pas celle qu'on croit : les deux catalogues parlent **la meme grammaire de verbes** (lire, ecrire, chercher, deplacer) et **aucun des deux ne porte le domaine** — le serveur de fichiers officiel ne connait ni le livre ni l'editeur, et le site non plus. Douze outils fichiers cote rayon, quarante-deux outils CMS cote site, pas un seul outil metier dans l'histoire : l'ambiguite que la suite mesure ne vient pas d'une symetrie de domaines, elle vient de **deux grammaires generiques qui se recouvrent** — deux casiers a la meme devanture, et la devanture ne dit meme pas ce qu'il y a dedans.


## 4. Premiere approche : le chevauchement structurel (Jaccard)

Avant de mesurer un agent, on peut tenter de classer le risque par la forme des deux catalogues. Chaque outil recoit une signature — sa classe de verbe (lister, lire, ecrire, modifier, supprimer) et son objet — puis on compare les signatures deux a deux par Jaccard. C'est l'approche de la version anterieure du carnet ; elle est rejouee ici sur les **deux catalogues reels** : les 42 outils du site contre les 12 outils du serveur filesystem.

La conclusion de cette section est volontairement provisoire : on verra qu'elle ne classe pas le risque.

### Ce que le format `(verbe, cible)` achete ici

Chaque outil est reduit a une signature : une **classe de verbe** (lister, lire, ecrire, modifier, supprimer) et un **objet**. Ce format est volontairement plus pauvre qu'une vraie fiche d'outil MCP, qui decrirait un schema d'arguments, des types de retour, des effets de bord et des conditions d'erreur. La pauvrete est le choix pedagogique : en reduisant chaque outil a ce qu'il *fait* (le verbe) et a *quoi* (l'objet), on rend deux catalogues comparables ligne a ligne, ce qu'une description riche rendrait incomparable.

C'est cette reduction qui rend possible la mesure de cette section : l'indice de Jaccard compare des **signatures**, pas des interfaces. Les mesures du carnet classent des *capacites*, pas des *schemas*.

Ce que le format ne voit pas, et que la lecture de cette section montre : la difference entre un verbe generique et un verbe metier, et surtout **l'origine** d'un outil — le site et le rayon n'apparaissent nulle part dans la signature. Jaccard ne peut donc pas, par construction, dire lequel des deux un agent choisira.


In [8]:
CLASSES_VERBES = {
    "list": "lister", "count": "lister", "tree": "lister",
    "get": "lire", "read": "lire", "search": "lire",
    "create": "ecrire", "add": "ecrire", "insert": "ecrire",
    "upload": "ecrire", "write": "ecrire",
    "update": "modifier", "alter": "modifier", "set": "modifier",
    "edit": "modifier", "move": "modifier",
    "delete": "supprimer", "remove": "supprimer",
}


def signature(nom):
    """(classe de verbe, objet) deduits du nom d'outil."""
    morceaux = nom.split("_")
    if morceaux and morceaux[0] in ("wp", "mwai"):
        morceaux = morceaux[1:]
    classe = next((c for p, c in CLASSES_VERBES.items()
                   if morceaux and morceaux[0] == p), "autre")
    objet = morceaux[-1] if morceaux else nom
    return classe, objet


def jaccard(jeu_a, jeu_b):
    union = set(jeu_a) | set(jeu_b)
    return len(set(jeu_a) & set(jeu_b)) / len(union) if union else 0.0


paires = []
for site in outils_exposes:
    classe_s, objet_s = signature(site["name"])
    for fiche_o in outils_rayon:
        classe_r, objet_r = signature(fiche_o["name"])
        coefficient = jaccard({classe_s, objet_s}, {classe_r, objet_r})
        if coefficient > 0:
            paires.append((round(coefficient, 3), site["name"], fiche_o["name"],
                           f"{classe_s}/{objet_s}", f"{classe_r}/{objet_r}"))

paires.sort(reverse=True)
print(f"{len(paires)} paires de coefficient > 0\n")
print(f"{'Jaccard':>7}  {'site':24s} {'rayon':26s} signatures")
for coefficient, site, rayon_o, sig_s, sig_r in paires[:12]:
    print(f"{coefficient:>7}  {site:24s} {rayon_o:26s} {sig_s} x {sig_r}")


101 paires de coefficient > 0

Jaccard  site                     rayon                      signatures
  0.333  wp_write_blocks          write_file                 ecrire/blocks x ecrire/file
  0.333  wp_write_blocks          create_directory           ecrire/blocks x ecrire/directory
  0.333  wp_upload_request        write_file                 ecrire/request x ecrire/file
  0.333  wp_upload_request        create_directory           ecrire/request x ecrire/directory
  0.333  wp_upload_media          write_file                 ecrire/media x ecrire/file
  0.333  wp_upload_media          create_directory           ecrire/media x ecrire/directory
  0.333  wp_update_user           move_file                  modifier/user x modifier/file
  0.333  wp_update_user           edit_file                  modifier/user x modifier/file
  0.333  wp_update_term           move_file                  modifier/term x modifier/file
  0.333  wp_update_term           edit_file                  modifier/term 

### Interpretation

Le signal est **plat** : toutes les paires qui partagent une classe de verbe tombent au meme coefficient — `wp_create_post` et `write_file` (les deux ecrivent) ressemblent a Jaccard autant que `wp_get_post` et `read_file` (les deux lisent). Et surtout, la signature ne porte **aucune trace de l'origine** : rien dans `ecrire/post` ou `ecrire/file` ne dit que le premier publie sur le site et le second ecrit dans le rayon. Jaccard mesure ce qui se ressemble **par la forme** ; il ne dit ni ce qu'un agent fera de la ressemblance, ni entre quels systemes il devra choisir. C'est precisement pour cela que la suite se mesure sur un modele.


## 5. L'aboutissement : le comportement d'un agent face aux deux catalogues

On fusionne les deux catalogues — les 42 fonctions du site plus l'outil de protocole `mcp_ping` (tel que `tools/list` le rend), et les 12 outils du rayon — et on les presente a un modele reel, avec une batterie de **seize taches**. Dix visent un systeme sans ambiguite : cinq le site, cinq le rayon. Six **traversent la frontiere**, ou la double ecriture est possible.

Trois precautions distinguent cette batterie d'un simple passage unique :

- **des repetitions** — chaque tache est posee `REPETITIONS` fois au meme modele (temperature non nulle) : un verdict isole ne veut rien dire, chaque taux est donc rendu avec son **intervalle de confiance de Wilson** ;
- **deux modeles** — la meme batterie passe sur deux moteurs de la meme famille mais de tailles differentes : si le petit rate ou le grand reussit, c'est un effet **modele** ; si les deux se trompent pareil, c'est un effet **protocole** ;
- **des descriptions coupees a 200 caracteres** — le catalogue fusionne (55 outils) serialise depasse la fenetre de 8192 tokens du petit moteur ; les descriptions sont coupees a l'identique pour les deux moteurs, pour que la comparaison reste equitable. C'est deja un enseignement : exposer 55 outils a un agent a un cout en contexte, avant meme le premier appel.

Chaque reponse est classee :

| Verdict | Sens |
|---|---|
| `CORRECT` | l'outil attendu a ete appele |
| `TRAVERSEE` | sur une tache frontiere, l'outil de l'autre systeme a ete appele — c'est l'evenement de double ecriture |
| `MAUVAIS_OUTIL` | un autre outil du meme systeme a ete appele |
| `AUCUN_OUTIL` | le modele a repondu en texte sans appeler d'outil |
| `HORS_CATALOGUE` | le modele a nomme un outil qui n'existe dans aucun des deux catalogues |

Deux de ces taches frontieres sont **explicites** : la consigne nomme le systeme visé, et le piege est de tomber sur l'outil homonyme de l'autre. *« Enregistre la fiche de lecture du livre X dans le rayon »* dit bien *fiche* et *rayon* (`write_file`), mais `wp_create_post` ecrit lui aussi ; *« Publie la fiche de lecture du livre X pour que les lecteurs la decouvrent »* dit *publie* (`wp_create_post`), mais la fiche mentionnee vit dans le rayon.

Quatre sont **ambigues par construction** : la consigne ne dit pas de quel systeme il s'agit, et un lecteur humain hesiterait lui aussi. *« Archive l'annonce du livre X »* se lit comme une trace rangee dans le rayon ou comme une page du site ; *« Recherche tout ce qui parle de neurones »* ne dit ni fiche ni article.

On fixe la convention d'arbitrage — **une fiche de lecture vit dans le rayon, une publication vit sur le site** — et on regarde si le modele respecte cette frontiere, s'il ecrit de l'autre cote, ou s'il renonce a appeler un outil. Ce n'est pas un piege : c'est precisement la situation que la double ecriture rend possible, et c'est la seule qu'un agent doit savoir regler.


In [9]:
from openai import OpenAI

# Le petit moteur a une fenetre de 8192 tokens : le catalogue fusionne serialise
# en entier (8364 tokens mesures) ne tient pas. On coupe les descriptions a 200
# caracteres, a l'identique pour les deux moteurs, pour garder la comparaison
# equitable -- et le prompt complet retombe a ~7400 tokens.
LIMITE_DESCRIPTION = 200


def schema_openai(nom, description, schema):
    """Fiche d'outil au format attendu par l'API chat/completions."""
    return {"type": "function",
            "function": {"name": nom,
                         "description": (description or "")[:LIMITE_DESCRIPTION],
                         "parameters": schema or {"type": "object", "properties": {}}}}


# Le site arrive par le protocole MCP (tools/list, section 3) : ses schemas
# d'arguments sont ceux que le serveur a negocies, pas une reconstruction.
outils_fusionnes = []
for outil in outils_consommes:
    outils_fusionnes.append(schema_openai(outil["name"], outil.get("description", ""),
                                          outil.get("inputSchema")))
for outil in outils_rayon:
    outils_fusionnes.append(schema_openai(outil["name"], outil["description"],
                                          outil["inputSchema"]))

noms_catalogue = {o["function"]["name"] for o in outils_fusionnes}
cote_site = {o["name"] for o in outils_consommes}
cote_rayon = {o["name"] for o in outils_rayon}
print("outils fusionnes :", len(outils_fusionnes),
      f"({len(cote_site)} site + {len(cote_rayon)} rayon)")
print("doublons de nom  :", len(cote_site) + len(cote_rayon) - len(noms_catalogue))
print("descriptions coupees a", LIMITE_DESCRIPTION, "caracteres (fenetre du petit moteur)")

outils fusionnes : 55 (43 site + 12 rayon)
doublons de nom  : 0
descriptions coupees a 200 caracteres (fenetre du petit moteur)


### Lecture

Deux nombres, deux lecons. **55 outils fusionnes** (43 du site + 12 du rayon) : les deux catalogues cohabitent dans un seul espace d'appel, celui que l'agent verra au prochain tour. **0 doublon de nom** : la coexistence ne fait aucun conflit syntaxique — `write_file` ne s'appelle jamais `wp_write_file`, `wp_create_post` ne s'appelle jamais `create_post`. C'est important pour la suite : l'ambiguite qu'on va mesurer ne pourra pas etre excusee par une collision de nommage ou un outillage confus. Chaque outil a un nom propre et non equivoque ; si l'agent se trompe de systeme, ce sera le **sens** du geste (ranger dans le rayon ou publier sur le site ?) qui l'aura induit en erreur, pas la taxonomie des outils. La troisieme ligne — descriptions coupees a **200 caracteres**, la fenetre du petit moteur — fixe le cadre : le banc mesure ce qu'un agent de 0,8 milliard de parametres peut lire, pas ce qu'un contexte long epuiserait.


In [10]:
BATTERIE = [
    # cote site : la consigne nomme le site, un seul outil s'impose
    ("Liste les extensions installees sur le site.", "wp_list_plugins", None),
    ("Montre-moi les utilisateurs du site.", "wp_get_users", None),
    ("Cree un article de blog intitule Bonjour avec un court contenu.", "wp_create_post", None),
    ("Supprime l'article numero 42.", "wp_delete_post", None),
    ("Change le titre du site dans les options.", "wp_update_option", None),
    # cote rayon : la consigne nomme le rayon, un seul outil s'impose
    ("Liste le contenu du rayon de fiches de lecture.", "list_directory", None),
    ("Montre le contenu de la fiche le-phare-de-valmont.md dans le rayon.", "read_file", None),
    ("Trouve dans le rayon les fichiers dont le nom contient jardins.", "search_files", None),
    ("Range la fiche jardins-suspendus.md dans le sous-dossier vieux du rayon.", "move_file", None),
    ("Quelle est la taille et la date de modification de la fiche le-phare-de-valmont.md ?",
     "get_file_info", None),
    # frontiere explicite : la consigne nomme le systeme, le piege est l'homonyme
    ("Enregistre la fiche de lecture du livre Intelligence artificielle et decision, "
     "auteur M. Aubert, dans le rayon.", "write_file", "wp_create_post"),
    ("Publie la fiche de lecture du livre Les Caprices de Valmont pour que les lecteurs "
     "la decouvrent.", "wp_create_post", "write_file"),
    # frontiere ambigue : la consigne ne dit pas de quel systeme il s'agit
    ("Archive l'annonce du livre Les reseaux de neurones.", "write_file", "wp_create_post"),
    ("Montre-moi les fiches qui parlent de neurones.", "search_files", "wp_get_posts"),
    ("Recherche tout ce qui parle de neurones.", "search_files", "wp_get_posts"),
    ("Corrige le titre du document les-reseaux-de-neurones.md.", "edit_file", "wp_alter_post"),
]
print(f"batterie : {len(BATTERIE)} taches "
      f"({sum(1 for _, _, r in BATTERIE if r is None)} ciblees, "
      f"{sum(1 for _, _, r in BATTERIE if r is not None)} frontieres)")


batterie : 16 taches (10 ciblees, 6 frontieres)


In [11]:
REPETITIONS = 25
TEMPERATURE = 0.7


def classer(attendu, obtenu):
    """Verdict : juste, abandon, hors catalogue, ou passage d'un systeme a l'autre."""
    if obtenu == attendu:
        return "CORRECT"
    if obtenu is None:
        return "AUCUN_OUTIL"
    if obtenu not in noms_catalogue:
        return "HORS_CATALOGUE"
    if (obtenu in cote_site) != (attendu in cote_site):
        return "TRAVERSEE"
    return "MAUVAIS_OUTIL"


def interroger(client, modele, prompt, systeme=None):
    messages = ([{"role": "system", "content": systeme}] if systeme else [])
    messages.append({"role": "user", "content": prompt})
    debut = time.time()
    reponse = client.chat.completions.create(
        model=modele,
        messages=messages,
        tools=outils_fusionnes,
        tool_choice="auto",
        temperature=TEMPERATURE,
        timeout=120,
    )
    choix = reponse.choices[0]
    nom = choix.message.tool_calls[0].function.name if choix.message.tool_calls else None
    return nom, choix.finish_reason, round(time.time() - debut, 1)


resultats = []  # une ligne par (moteur, tache, repetition)
for nom_moteur, moteur in MOTEURS.items():
    client = OpenAI(base_url=moteur["url"], api_key=moteur["cle"])
    print(f"=== moteur [{nom_moteur}] : {moteur['modele']} ===", flush=True)
    for prompt, attendu, risque in BATTERIE:
        verdicts_tache = []
        for _ in range(REPETITIONS):
            obtenu, fin, duree = interroger(client, moteur["modele"], prompt)
            verdict = classer(attendu, obtenu)
            verdicts_tache.append(verdict)
            resultats.append({"moteur": nom_moteur, "modele": moteur["modele"],
                              "prompt": prompt, "attendu": attendu, "risque": risque,
                              "obtenu": obtenu, "verdict": verdict,
                              "fin": fin, "duree_s": duree})
        resume = "  ".join(f"{v}={verdicts_tache.count(v)}"
                           for v in sorted(set(verdicts_tache), reverse=True))
        piege = f"  [piege : {risque}]" if risque else ""
        print(f"  {prompt[:52]:52s} {resume}{piege}", flush=True)

Il faut toutefois garder en tête ce que le format ne voit pas, parce que la conclusion finale
en dépend. Deux outils de même signature ne sont pas interchangeables pour autant : un
`lire livre` interne qui sert le texte déjà publié dans le site et un `lire livre` externe qui
interroge une base bibliographique font bien « la même chose » au sens de la signature, mais
diffèrent par la fraîcheur des données, le coût de l'appel, la latence, la disponibilité. La
signature est l'unité de mesure du *recouvrement*, pas du *remplacement* — elle dit « vous
possédez déjà cette capacité », jamais « inutile d'appeler l'externe pour cela ». C'est une
distinction que la règle de canonical de l'exemple guidé 3 sera chargée d'opérationnaliser.

=== moteur [principal] : qwen35-0.8b ===


  Liste les extensions installees sur le site.         CORRECT=25


  Montre-moi les utilisateurs du site.                 HORS_CATALOGUE=3  CORRECT=22


  Cree un article de blog intitule Bonjour avec un cou TRAVERSEE=7  MAUVAIS_OUTIL=8  CORRECT=10


  Supprime l'article numero 42.                        TRAVERSEE=1  MAUVAIS_OUTIL=1  CORRECT=23


  Change le titre du site dans les options.            MAUVAIS_OUTIL=20  HORS_CATALOGUE=2  CORRECT=2  AUCUN_OUTIL=1


... (25 outputs left out)

  Montre-moi les fiches qui parlent de neurones.       TRAVERSEE=25  [piege : wp_get_posts]


  Recherche tout ce qui parle de neurones.             TRAVERSEE=14  CORRECT=9  AUCUN_OUTIL=2  [piege : wp_get_posts]


  Corrige le titre du document les-reseaux-de-neurones MAUVAIS_OUTIL=25  [piege : wp_alter_post]



lignes collectees : 800


In [12]:
import math


def intervalle_wilson(k, n, z=1.96):
    """Proportion observee et intervalle de confiance a 95 % (Wilson)."""
    if n == 0:
        return 0.0, 0.0, 0.0
    p = k / n
    denom = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / denom
    demi = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return p, centre - demi, centre + demi


frontieres_idx = [i for i, (_, _, r) in enumerate(BATTERIE) if r is not None]
taches_frontieres = {BATTERIE[i][0] for i in frontieres_idx}

for nom_moteur in MOTEURS:
    lignes = [l for l in resultats if l["moteur"] == nom_moteur]
    modele = lignes[0]["modele"] if lignes else "?"
    print(f"=== moteur [{nom_moteur}] : {modele} — {len(lignes)} reponses "
          f"({len(BATTERIE)} taches x {REPETITIONS}) ===")
    for verdict in ("CORRECT", "TRAVERSEE", "MAUVAIS_OUTIL", "AUCUN_OUTIL", "HORS_CATALOGUE"):
        k = sum(1 for l in lignes if l["verdict"] == verdict)
        if not k:
            continue
        p, bas, haut = intervalle_wilson(k, len(lignes))
        print(f"  {verdict:13s} {k:4d}/{len(lignes)}  {100 * p:5.1f} %  "
              f"IC95 [{100 * bas:.1f} ; {100 * haut:.1f}]")

    # Le taux qui interesse la double ecriture : la traverse sur les seules
    # taches frontieres (16 x REPETITIONS reponses ne seraient pas comparables).
    lignes_f = [l for l in lignes if l["prompt"] in taches_frontieres]
    k_t = sum(1 for l in lignes_f if l["verdict"] == "TRAVERSEE")
    p, bas, haut = intervalle_wilson(k_t, len(lignes_f))
    print(f"  -> traversees sur frontieres : {k_t}/{len(lignes_f)}  "
          f"{100 * p:.1f} %  IC95 [{100 * bas:.1f} ; {100 * haut:.1f}]")
    for prompt in sorted(taches_frontieres):
        vf = [l["verdict"] for l in lignes if l["prompt"] == prompt]
        ktf = vf.count("TRAVERSEE")
        pt, bt, ht = intervalle_wilson(ktf, len(vf))
        print(f"     {prompt[:56]:56s} {ktf}/{len(vf)}  "
              f"{100 * pt:4.0f} %  IC95 [{100 * bt:.0f} ; {100 * ht:.0f}]")
    print()


=== moteur [principal] : qwen35-0.8b — 400 reponses (16 taches x 25) ===
  CORRECT        130/400   32.5 %  IC95 [28.1 ; 37.2]
  TRAVERSEE      101/400   25.2 %  IC95 [21.2 ; 29.7]
  MAUVAIS_OUTIL  151/400   37.8 %  IC95 [33.1 ; 42.6]
  AUCUN_OUTIL      6/400    1.5 %  IC95 [0.7 ; 3.2]
  HORS_CATALOGUE   12/400    3.0 %  IC95 [1.7 ; 5.2]
  -> traversees sur frontieres : 90/150  60.0 %  IC95 [52.0 ; 67.5]
     Archive l'annonce du livre Les reseaux de neurones.      21/25    84 %  IC95 [65 ; 94]
     Corrige le titre du document les-reseaux-de-neurones.md. 0/25     0 %  IC95 [0 ; 13]
     Enregistre la fiche de lecture du livre Intelligence art 12/25    48 %  IC95 [30 ; 67]
     Montre-moi les fiches qui parlent de neurones.           20/25    80 %  IC95 [61 ; 91]
     Publie la fiche de lecture du livre Les Caprices de Valm 12/25    48 %  IC95 [30 ; 67]
     Recherche tout ce qui parle de neurones.                 25/25   100 %  IC95 [87 ; 100]

=== moteur [contraste] : qwen35-4b — 400

### Lecture

**La confusion fréquente** : un administrateur qui veut « connecter un outil
MCP » ne sait pas s'il doit l'ajouter côté serveur (exposer un nouvel outil
Valmont aux agents externes) ou côté client (consommer un outil externe depuis
Valmont). La réponse dépend du **sens du fil** : qui appelle qui ?


#### Lecture de la sortie : la symétrie 7 / 7 et le vocabulaire partagé

La sortie ne montre que deux compteurs — `7 outils` servis, `7 outils` appelés — mais cette
symétrie apparente est déjà une information : le site n'est ni dominé par sa consommation ni
réduit à son offre. Le déséquilibre serait en soi un signal architectural. Un site qui expose
dix fois ce qu'il consomme est surtout un serveur, dont la valeur est dans ce que le monde
peut en tirer. Un site qui consomme dix fois ce qu'il expose est surtout un client, dont la
valeur est dans ce qu'il saucissonne derrière ses appels. Le rapport un-pour-un de Valmont
dessine un profil d'intégration équilibrée : le site s'appuie autant qu'il sert — ce qui ne
préjuge en rien de la *qualité* de chaque côté, seulement de sa physionomie.

Plus significatif encore : les deux catalogues, écrits l'un pour l'offre et l'autre pour la
dépendance, partagent une grande partie de leur vocabulaire de verbes — `lire`, `chercher`,
`modifier`, `soumettre` apparaissent des deux côtés, comme les cellules suivantes vont le
mesurer explicitement. Ce partage n'est pas une négligence de modélisation : c'est la trace du
fait que les deux côtés parlent du même métier — les livres, leurs comptes-rendus, leurs
fiches — chacun avec sa perspective. Et c'est précisément ce vocabulaire commun qui rendra le
chevauchement de la section 4 *possible* : si les verbes étaient disjoints entre les deux
catalogues, la question du recouvrement ne se poserait même pas, et l'indice de la section 5
vaudrait zéro par construction.

Retenir le geste de lecture, car il est réutilisable partout : deux compteurs égaux ne disent
pas « deux catalogues équivalents ». Ils disent « deux ensembles de même taille » — rien sur
le contenu. Une égalité de cardinalité est compatible avec l'identité parfaite comme avec
l'intersection vide. La comparaison de contenu exige l'outillage ensembliste que le notebook
construit dans les sections suivantes ; c'est exactement la distance entre *compter* et
*comparer*.
Une précision d'ordre de grandeur, pour fixer les idées sur ce que ces compteurs *pèsent* :
sept outils de chaque côté, c'est la taille d'un catalogue pédagogique, lisible en une
screen. Les mesures qui suivent seraient exactement les mêmes sur des catalogues de soixante
dix outils — mais la lecture à l'œil, elle, ne survivrait pas : c'est la raison pour laquelle
le notebook s'outille dès maintenant (signatures, indice, différence) au lieu de compter sur
l'inspection directe. La taille présente rend chaque étape vérifiable à la main ; la méthode
restera la seule viable quand la taille ne le permettra plus.

## 6. Lecture croisee : la forme predit-elle le comportement ?

Si Jaccard classait le risque, les traversees observees suivraient les paires les mieux classees. On confronte les deux mesures : pour chaque paire (outil obtenu, outil attendu) qui a produit une traversee — toutes repetitions et tous moteurs confondus — on affiche le nombre d'occurrences et le coefficient structurel calcule en section 4.


In [13]:
index_jaccard = {}
for coefficient, site, rayon_o, _, _ in paires:
    index_jaccard[(site, rayon_o)] = max(index_jaccard.get((site, rayon_o), 0), coefficient)
    index_jaccard[(rayon_o, site)] = max(index_jaccard.get((rayon_o, site), 0), coefficient)

# Toutes les traversees, toutes repetitions et moteurs confondus.
paires_traversees = {}
for ligne in resultats:
    if ligne["verdict"] == "TRAVERSEE":
        cle = (ligne["obtenu"], ligne["attendu"])
        paires_traversees[cle] = paires_traversees.get(cle, 0) + 1

print("Traversees observees et coefficient structurel de la paire :")
if not paires_traversees:
    print("  (aucune traversee sur cette execution)")
for (obtenu, attendu), nombre in sorted(paires_traversees.items(), key=lambda kv: -kv[1]):
    coefficient = index_jaccard.get((obtenu, attendu), 0.0)
    print(f"  - {obtenu} vs {attendu} : {nombre} fois, Jaccard {coefficient}")

print()
coef_max = paires[0][0] if paires else 0
print(f"paires au coefficient maximal ({coef_max}) :",
      sum(1 for c, *_ in paires if c == coef_max))
print("Conclusion provisoire : la forme ne classe pas le risque — "
      "toutes les paires ecriture/lecture valent le meme coefficient, "
      "seule la mesure de comportement revele lesquelles traverse.")


Traversees observees et coefficient structurel de la paire :
  - wp_get_posts vs search_files : 39 fois, Jaccard 0.333
  - wp_get_posts vs write_file : 29 fois, Jaccard 0.0
  - wp_list_plugins vs search_files : 22 fois, Jaccard 0.0
  - read_file vs wp_create_post : 13 fois, Jaccard 0.0
  - wp_list_block_patterns vs search_files : 13 fois, Jaccard 0.0
  - wp_list_plugins vs write_file : 12 fois, Jaccard 0.0
  - mcp_ping vs search_files : 9 fois, Jaccard 0.0
  - mcp_ping vs write_file : 8 fois, Jaccard 0.0
  - write_file vs wp_create_post : 6 fois, Jaccard 0.333
  - wp_get_posts vs list_directory : 5 fois, Jaccard 0.0
  - mcp_ping vs list_directory : 4 fois, Jaccard 0.0
  - wp_get_post vs write_file : 4 fois, Jaccard 0.0
  - wp_list_block_patterns vs write_file : 4 fois, Jaccard 0.0
  - wp_get_post_types vs list_directory : 3 fois, Jaccard 0.0
  - wp_list_block_patterns vs list_directory : 2 fois, Jaccard 0.333
  - wp_create_post vs write_file : 2 fois, Jaccard 0.333
  - read_file vs wp_

### Lecture

La confrontation est sans appel, et plus rude pour la forme que ne le laissait croire le piege. La paire la plus frequente — `wp_get_posts` vs `search_files`, **39 fois** — porte bien le Jaccard maximal **0,333** ; mais la deuxieme (`wp_get_posts` vs `write_file`, 29 fois), la troisieme (`wp_list_plugins` vs `search_files`, 22 fois) et la grande majorite du trafic observe sont a **Jaccard 0,0** : **130 des 179 traversees** observees se produisent sur des paires sans aucune signature commune, invisibles a la metrique. Autrement dit, le signal structurel n'explique ni l'ordre ni la densite : les 101 paires au coefficient maximal ne sont pas celles qu'on observe, et les traversees les plus frequentes sont precisement celles que la forme ne voit pas. Aucun seuil pose sur cette metrique n'attrape le risque sans attraper tout le reste avec lui. La conclusion provisoire de la cellule devient definitive sur ce banc : seule la mesure de comportement revele la frontiere.


## 7. Exemples guides

### Exemple 1 — relire un choix non correct

Une traversee n'est pas une panne : c'est un choix d'agent qui ecrit dans le mauvais systeme. Un abandon en est une autre : l'agent a repondu en texte, sans appeler aucun outil. Les deux ne demandent pas la meme correction — et le tableau de la section 5 dit seulement combien de chaque ; ici, on en **relit un** dans le detail, moteur et repetition compris. Le code distingue les cas, car la phrase affichee doit decrire ce qui s'est reellement passe.


In [14]:
# Exemple guide 1 : le detail d'un choix non correct, moteur et repetition compris.
candidats = [l for l in resultats if l["verdict"] != "CORRECT"]
evenement = candidats[0] if candidats else resultats[-1]
print("moteur   :", evenement["moteur"], "(", evenement["modele"], ")")
print("prompt   :", evenement["prompt"])
print("attendu  :", evenement["attendu"])
print("obtenu   :", evenement["obtenu"])
print("verdict  :", evenement["verdict"])
print("fin      :", evenement["fin"], "| duree :", evenement["duree_s"], "s")
print()
if evenement["verdict"] == "TRAVERSEE":
    print("Lecture : l'agent a appele", evenement["obtenu"],
          "la ou la tache demandait", evenement["attendu"],
          "— il a ecrit dans le mauvais systeme.")
elif evenement["verdict"] == "AUCUN_OUTIL":
    print("Lecture : l'agent n'a appele aucun outil et a repondu en texte.",
          "Aucune ecriture n'a donc traverse la frontiere ;",
          "l'echec mesure ici est l'abandon, pas la traversee.")
elif evenement["verdict"] == "MAUVAIS_OUTIL":
    print("Lecture : l'agent a appele", evenement["obtenu"],
          "au lieu de", evenement["attendu"],
          "— mauvais outil, mais du bon systeme : rien n'a traverse.")
else:
    print("Lecture : l'agent a appele", evenement["obtenu"],
          "pour la tache attendue", evenement["attendu"], "— l'ecart porte sur autre chose.")


moteur   : principal ( qwen35-0.8b )
prompt   : Montre-moi les utilisateurs du site.
attendu  : wp_get_users
obtenu   : wp_list_users
verdict  : HORS_CATALOGUE
fin      : tool_calls | duree : 0.3 s

Lecture : l'agent a appele wp_list_users pour la tache attendue wp_get_users — l'ecart porte sur autre chose.


### Lecture

Les signatures `(lire, livre)` et `(chercher, livre)` sont présentes **des deux
côtés**. Concrètement : un chatbot Valmont qui veut lire une fiche livre a deux
chemins — `valmont_get_book` (interne) ou `bookgraph_get_book_info` (externe).
Lequel choisir ? S'ils divergent (données différentes, fraîcheur différente),
l'agent peut donner une réponse incohérente selon le chemin. S'ils
convergent, l'un des deux est **redondant**.


#### Ce que la concentration du chevauchement révèle

La sortie classe les faits mieux qu'un discours : les trois signatures communes —
`chercher livre`, `lire livre`, `modifier livre` — sont toutes trois centrées sur la même
entité, le livre. Ce n'est pas un hasard d'échantillonnage, c'est la structure du métier :
le livre est l'objet que les deux sites manipulent nécessairement, celui autour duquel
l'offre interne et la dépendance externe se rencontrent. Le recouvrement se concentre là où
les deux univers d'action se superposent — et il serait resté concentré sur d'autres entités
communes si le métier en avait : c'est une loi de lecture, pas une particularité du jeu de
données.

La lecture duale est tout aussi instructive : ce que chaque côté garde en propre dessine sa
spécialisation. Côté exposé, les signatures exclusives sont éditoriales — `lire manuscrit`,
`soumettre manuscrit`, `assigner lecteur`, `lire compte-rendu` : le flux de fabrication d'un
texte, de la soumission brute jusqu'au compte-rendu publié, ce que Valmont sait faire de bout
en bout. Côté consommé, les exclusives sont bibliographiques — `enrichir livre`, `lier livre`,
`lire auteur`, `soumettre compte-rendu` : la fiche du livre relâchée dans un graphe de
connaissances, ce que bookgraph sait faire de ce que Valmont fabrique. L'interne gère la
*fabrication*, l'externe enrichit la *connaissance* du produit fabriqué — et les quatre
exclusives de chaque côté, en nombre égal, disent une dépendance mutuelle plutôt qu'une
subordination.

Cette lecture prépare la suite de deux manières. D'abord, le chevauchement n'est pas
uniformément distribué sur les catalogues : il a une géographie. Quantifier son ampleur
globale (section 5) ne dispensera jamais de regarder *où* il se situe — car un recouvrement
sur les verbes de lecture se juge différemment d'un recouvrement sur les verbes d'écriture :
lire deux fois coûte une requête, écrire deux fois coûte une incohérence. C'est cette
distinction que l'exemple guidé 2 fera travailler sur le sous-ensemble des verbes d'écriture.
Ensuite, la spécialisation des exclusives annonce la valeur du branchement (section 6) :
l'apport réel de l'externe se lira dans ses signatures exclusives, pas dans ses doublons.

### Exemple 2 — des ecritures reelles, des deux cotes, par le protocole

La batterie mesure des **choix** ; cet exemple montre que les outils choisis ecrivent vraiment. Deux allers-retours complets : cote site, un brouillon cree par `tools/call`, relu, puis supprime ; cote rayon, une fiche ecrite par le serveur filesystem puis relue dans la meme session. Chaque systeme garde l'etat attendu apres le passage — c'est la contre-verification de fin.


In [15]:
# Exemple guide 2a : cote site, un brouillon cree, relu, supprime — par MCP HTTP.
import re

rep_create = mcp_rpc("tools/call", {"name": "wp_create_post", "arguments": {
    "post_title": "Article ephemere du carnet consommer-vs-exposer",
    "post_content": "Cree par un tools/call MCP, relu puis supprime dans la meme demonstration.",
    "post_status": "draft",
}})
message = texte_outil(rep_create)
print("wp_create_post ->", message[:90])
article_id = int(re.search(r"ID (\d+)", message).group(1))

rep_get = mcp_rpc("tools/call", {"name": "wp_get_post", "arguments": {"ID": article_id}})
post = json.loads(texte_outil(rep_get))
print("wp_get_post    ->", post["post_title"], "| statut :", post["post_status"])

rep_del = mcp_rpc("tools/call", {"name": "wp_delete_post", "arguments": {"ID": article_id}})
print("wp_delete_post ->", texte_outil(rep_del)[:90])

comptage = json.loads(texte_outil(mcp_rpc("tools/call", {"name": "wp_count_posts",
                                                         "arguments": {}})))
print("verification   : drafts restants =", comptage["draft"], "(l'ephemere a disparu)")


wp_create_post -> Post created ID 28


wp_get_post    -> Article ephemere du carnet consommer-vs-exposer | statut : draft


wp_delete_post -> Post #28 deleted
verification   : drafts restants = 0 (l'ephemere a disparu)


# Exemple guide 2b : cote rayon, une fiche ecrite puis relue — session stdio filesystem.
FICHE_ESSAI = RAYON / "essai-ecriture.md"


async def ecrire_puis_relire_rayon():
    parametres = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", str(RAYON)],
    )
    async with stdio_client(parametres, errlog=JOURNAL_FS) as (lecture, ecriture):
        async with ClientSession(lecture, ecriture) as session:
            await session.initialize()
            ecrit = await session.call_tool("write_file", {
                "path": str(FICHE_ESSAI),
                "content": "# Essai d'ecriture\n\n"
                           "Fiche ecrite par un tools/call au serveur filesystem, "
                           "puis relue dans la meme session.\n",
            })
            relu = await session.call_tool("read_file", {"path": str(FICHE_ESSAI)})
            apres = await session.call_tool("list_directory", {"path": str(RAYON)})
            return ecrit.content[0].text, relu.content[0].text, apres.content[0].text


ecrit_fs, relu_fs, apres_fs = executer(ecrire_puis_relire_rayon())
print("write_file    ->", ecrit_fs[:80].replace("\n", " "))
print("read_file     ->", relu_fs.splitlines()[0], "/",
      len(relu_fs), "caracteres relus")
print("verification  -> le rayon contient desormais",
      len(apres_fs.splitlines()), "entrees (fiches initiales + essai-ecriture.md)")


#### Ce que l'indice ne voit pas : signatures contre outils

Le 0,27 porte sur des *signatures*, pas sur des outils — et l'écart entre les deux niveaux
n'est pas un abus de langage mais une distinction de mesure, invisible ici par coïncidence.
Le catalogue exposé compte sept outils pour sept signatures : chaque outil interne vise une
cible distincte, donc la projection outils-vers-signatures est ici une bijection. Le côté
consommé présente la même coïncidence. Mais rien ne garantit cette identité en général :
trois outils internes ciblant le livre avec le même verbe `lire` — par recherche, par
identifiant, par collection — compteraient pour *une seule* signature dans l'indice.
L'indice mesure le recouvrement des capacités, pas la redondance des implémentations.

Cette distinction devient opérationnelle dans l'exemple guidé 1, qui redescend précisément au
niveau des outils : parmi les outils *consommés*, lesquels sont redondants au sens des
signatures ? La mesure par signatures peut rester faible pendant qu'une masse d'outils
individuels duplique l'offre interne — si l'externe expose cinq variantes de recherche de
livre, elles ne pèsent qu'une signature dans l'union, mais cinq dépendances dans la
maintenance : cinq schémas d'arguments à suivre en version, cinq points de panne possibles,
cinq entrées de catalogue à auditer. Le coût réel d'une intégration se paie en outils, pas
en signatures ; la décision de brancher se prend en signatures, l'inventaire de ce qu'on
retire se fait en outils.

Règle de lecture en deux temps, que le reste du notebook applique et que l'exemple guidé 1
met en œuvre : les signatures pour décider *si* le recouvrement est significatif —
comparaison en espace commun, ratio borné, seuil écrit ; les outils pour décider *quoi*
retirer ou garder une fois la décision prise — inventaire nommé, entrée par entrée, avec
les noms d'outils réels du catalogue consommé. Les deux niveaux répondent à des questions
différentes et aucun ne se déduit mécaniquement de l'autre : c'est la raison pour laquelle
le notebook mesure aux deux, au lieu de choisir.
On peut le voir sur les données du notebook elles-mêmes, sans hypothèse externe : la sortie
de la section 4 imprime *sept* signatures pour le catalogue exposé, et l'inventaire des
sources en compte *sept* outils — mais rien dans la construction n'imposait l'égalité, et le
jour où Valmont ajoute `valmont_list_books_by_collection` à côté de `valmont_search_catalog`
et `valmont_get_book` (trois outils, la même signature `chercher livre` au sens large), le
compte de signatures ne bougera pas d'une unité alors que le catalogue aura grandi d'un
tiers. Toute lecture de l'indice devrait donc être accompagnée, dans un audit réel, du ratio
outils/signatures de chaque côté — un catalogue dont le ratio grimpe est un catalogue qui
s'étoffe en profondeur sur ses capacités existantes, pas un catalogue qui s'élargit.

### Exemple 3 — une politique de routage change-t-elle le choix ?

Si les traversees viennent d'une ambiguite de consigne, une politique explicite — quel serveur tient quel systeme — devrait les reduire. On ne le presume pas : sur le moteur principal, on rejoue les six taches frontieres autant de fois **sans** puis **avec** cette politique, et l'on compare les taux de traversee avec leurs intervalles. Une consigne peut ne rien changer — sur un petit modele, elle peut aussi ne pas etre lue — et c'est une des mesures a faire.


In [17]:
# Exemple guide 3 : six taches frontieres, sans puis avec politique de routage,
# sur le moteur principal, REPETITIONS_EX3 fois par bras.
POLITIQUE = (
    "Tu disposes de deux serveurs MCP. Les outils wp_* et mwai_* pilotent le site "
    "WordPress de la Maison Valmont : tout ce qui est publie ou modifie pour etre lu "
    "par les visiteurs du site passe par eux. Les autres outils (read_file, write_file, "
    "edit_file, move_file, search_files, list_directory, ...) tiennent le rayon de "
    "fiches de lecture : une fiche de lecture est un fichier du rayon, une publication "
    "est un article du site. Choisis le serveur du systeme que la tache modifie."
)

REPETITIONS_EX3 = 8
nom_p = next(iter(MOTEURS))
moteur_p = MOTEURS[nom_p]
client_p = OpenAI(base_url=moteur_p["url"], api_key=moteur_p["cle"])
print(f"moteur : [{nom_p}] {moteur_p['modele']} — {REPETITIONS_EX3} repetitions par bras\n")


def taux_resume(verdicts):
    return (f"{verdicts.count('CORRECT')}C/{verdicts.count('TRAVERSEE')}T/"
            f"{verdicts.count('MAUVAIS_OUTIL')}M/{verdicts.count('AUCUN_OUTIL')}A")


frontieres_ex3 = [(p, a, r) for p, a, r in BATTERIE if r is not None]
tous_sans, tous_avec = [], []
for prompt, attendu, risque in frontieres_ex3:
    sans_v, avec_v = [], []
    for _ in range(REPETITIONS_EX3):
        obtenu_s, _, _ = interroger(client_p, moteur_p["modele"], prompt)
        obtenu_a, _, _ = interroger(client_p, moteur_p["modele"], prompt, systeme=POLITIQUE)
        sans_v.append(classer(attendu, obtenu_s))
        avec_v.append(classer(attendu, obtenu_a))
    tous_sans.extend(sans_v)
    tous_avec.extend(avec_v)
    print(f"{prompt[:56]:56s} attendu {attendu}")
    print(f"  sans : {taux_resume(sans_v):16s} avec : {taux_resume(avec_v)}", flush=True)

print()
for etiquette, vs in (("sans politique", tous_sans), ("avec politique", tous_avec)):
    k_t = vs.count("TRAVERSEE")
    p_t, bas, haut = intervalle_wilson(k_t, len(vs))
    print(f"{etiquette:15s}: traversees {k_t}/{len(vs)}  {100 * p_t:5.1f} %  "
          f"IC95 [{100 * bas:.1f} ; {100 * haut:.1f}]")


moteur : [principal] qwen35-0.8b — 8 repetitions par bras



Enregistre la fiche de lecture du livre Intelligence art attendu write_file
  sans : 0C/5T/3M/0A      avec : 0C/0T/8M/0A


Publie la fiche de lecture du livre Les Caprices de Valm attendu wp_create_post
  sans : 0C/1T/6M/1A      avec : 0C/7T/1M/0A


Archive l'annonce du livre Les reseaux de neurones.      attendu write_file
  sans : 0C/4T/4M/0A      avec : 0C/3T/5M/0A


Montre-moi les fiches qui parlent de neurones.           attendu search_files
  sans : 0C/7T/0M/0A      avec : 1C/5T/0M/0A


Recherche tout ce qui parle de neurones.                 attendu search_files
  sans : 1C/7T/0M/0A      avec : 0C/8T/0M/0A


Corrige le titre du document les-reseaux-de-neurones.md. attendu edit_file
  sans : 0C/0T/8M/0A      avec : 0C/0T/8M/0A



sans politique : traversees 24/48   50.0 %  IC95 [36.4 ; 63.6]
avec politique : traversees 23/48   47.9 %  IC95 [34.5 ; 61.7]


### Lecture

Le verdict est en deux temps. **Au niveau global**, le chevauchement est
faible (Jaccard ~0,27) : sur 7 outils consommés, 4 apportent une signature que
l'interne n'a pas (`(enrichir, livre)`, `(lier, livre)`, `(lire, auteur)`,
`(soumettre, compte-rendu)`). Le branchement est donc justifié — il élargit
vraiment le champ fonctionnel.

**Mais un chevauchement global faible ne dispense pas d'examiner le
sous-ensemble écriture.** Parmi les 3 signatures communes, l'une est
`(modifier, livre)` — présente côté exposé (`valmont_update_book_meta`) **et**
côté consommé (`bookgraph_update_book`). C'est un **double-écrit** : si
l'agent met à jour la même fiche par les deux chemins, les deux sources
divergent. Un seul point de chevauchement sur un verbe d'écriture suffit à
créer un risque, même noyé dans un catalogue par ailleurs peu redondant.
C'est l'objet de l'exemple guidé 2.

**C'est la question que ce notebook formalise** : *faut-il brancher ce serveur
MCP externe ?* se répond en deux dénombrements — global (le chevauchement
total dit si le branchement vaut le coup) et fin (le chevauchement sur les
verbes d'écriture dit où est le risque résiduel).

#### Quatre signatures neuves : la dépendance jugée sur ce qu'elle apporte

La mesure finale inverse la perspective des précédentes : au lieu de compter ce qui se
recouvre, elle isole ce que le branchement *apporte*. La sortie en dresse la liste exacte —
`enrichir livre`, `lier livre`, `lire auteur`, `soumettre compte-rendu` — et chacune se lit
comme une capacité que Valmont ne possède pas dans son offre : enrichir une fiche de
relations bibliographiques, relier deux livres entre eux, consulter la fiche d'un auteur,
publier un avis vers l'extérieur. Quatre dépendances nouvelles, mais surtout quatre
capacités nouvelles — la nuance est tout le sujet de la mesure.

La lecture la plus utile n'est pas la liste mais sa *cohérence*. Les quatre signatures
neuves sont précisément les capacités d'un graphe de connaissances — enrichir les nœuds,
créer les arêtes, consulter les voisins (`lier` relie des livres, `lire auteur` consulte
le nœud adjacent, `enrichir` complète l'attributage). La dépendance externe n'apporte pas
quatre outils disparates, elle apporte *une fonctionnalité cohérente* : l'inscription du
catalogue Valmont dans un graphe plus large que lui. C'est cette cohérence qui fonde la
valeur du branchement bien davantage que le compte de quatre — un apport de quatre
signatures sans lien mutuel vaudrait moins qu'un apport d'une seule signature qui ouvre
une dimension. La décision d'intégration juge des *dimensions ouvertes*, pas des lignes
ajoutées.

Noter enfin la rigueur du geste de mesure, car c'est lui qui établit la preuve. Le compte
brut des outils consommés (sept) laissait la valeur *croire* acquise ; l'indice de Jaccard
(0,27) la laissait *deviner* favorable ; seule la différence `consommé − exposé` la
*démontre*, entrée par entrée, avec le nom de l'outil porteur pour chaque signature. Trois
mesures, trois certitudes croissantes — et la progression n'est pas décorative : chaque
mesure répond à une question que la précédente ne pouvait pas fermer. C'est la démarche,
plus encore que les chiffres, que les exemples guidés et les exercices suivants demandent
de savoir refaire.

## 7. Exemples guidés et exercices

Les trois **exemples guidés** qui suivent ont été résolus par un groupe d'étudiants de
l'EPF lors du TP2 de la session 2026-2027 (rendu
[#20558](https://github.com/jsboige/CoursIA/pull/20558)). Leur code est conservé tel
quel : il transforme les mesures des sections 4 à 6 en décisions, et ses fonctions
servent d'appui aux trois **exercices** placés après eux.

Les exercices restent à compléter : chaque stub renvoie `None`, la cellule affiche alors
« Exercice a completer », et le notebook s'exécute de bout en bout tant qu'ils ne sont
pas résolus.

### Exemple guidé 1 — la redondance par outil

Pour chaque outil du catalogue consommé, la fonction `partitionner_redondance` indique
s'il est **redondant** (sa signature existe déjà dans le catalogue exposé) ou
**nouveau**. Elle renvoie deux listes : les noms d'outils redondants et les noms
d'outils nouveaux.

In [18]:
def partitionner_redondance(expose, consomme):
    """Renvoie (redondants, nouveaux) -- listes de noms d'outils du consomme.

    Un outil consomme est redondant si sa signature existe deja cote expose.
    """
    signatures_exposees = signatures_catalogue(expose)
    redondants = []
    nouveaux = []
    for nom, spec in consomme.items():
        if signature(spec) in signatures_exposees:
            redondants.append(nom)
        else:
            nouveaux.append(nom)
    return redondants, nouveaux

redondants, nouveaux = partitionner_redondance(catalogue_expose, catalogue_consomme)
print("Redondants :", redondants)
print("Nouveaux    :", nouveaux)


Redondants : ['bookgraph_get_book_info', 'bookgraph_search_books', 'bookgraph_update_book']
Nouveaux    : ['bookgraph_enrich_book', 'bookgraph_get_author', 'bookgraph_link_similar', 'bookgraph_submit_review']


### Exemple guidé 2 — le coût d'un double-écrit

Un outil redondant peut causer un **double-écrit** : si l'agent met à jour une
fiche livre via `valmont_update_book_meta` (interne) puis via un outil externe
équivalent, les deux sources divergent. La fonction `detecter_double_ecrit` renvoie
les signatures `(verbe, cible)` présentes des deux côtés **avec un verbe d'écriture**
(`modifier`, `soumettre`, `enrichir`, `lier`…), c'est-à-dire le sous-ensemble du
chevauchement qui est dangereux (lecture redondante = tolérable, écriture redondante =
risque).

In [19]:
VERBES_ECRITURE = {"modifier", "soumettre", "enrichir", "lier", "assigner", "creer", "supprimer"}

def detecter_double_ecrit(expose, consomme):
    """Renvoie les signatures (verbe, cible) presentes des deux cotes
    ET dont le verbe est un verbe d'ecriture -- le sous-ensemble dangereux
    du chevauchement.
    """
    signatures_communes = signatures_catalogue(expose) & signatures_catalogue(consomme)
    return {sig for sig in signatures_communes if sig[0] in VERBES_ECRITURE}

print("Double-écritures :", detecter_double_ecrit(catalogue_expose, catalogue_consomme))


Double-écritures : {('modifier', 'livre')}


### Exemple guidé 3 — choisir le canonical

Pour chaque signature commune, il faut désigner un **canonical** (l'outil à
utiliser par défaut). La fonction `choisir_canonical` renvoie, étant donné les deux
catalogues et une signature commune, le nom d'outil canonique selon une règle simple :
*préférer l'outil interne* (préfixe `valmont_`) sauf si l'externe est le seul à offrir
la signature.

In [20]:
def choisir_canonical(expose, consomme, sig_commune):
    """Pour une signature commune, renvoie le nom d'outil canonique.

    Regle : preferer l'outil interne (valmont_*); si aucun interne n'offre
    cette signature, prendre l'externe.
    """
    for nom, spec in expose.items():
        if signature(spec) == sig_commune:
            return nom
    for nom, spec in consomme.items():
        if signature(spec) == sig_commune:
            return nom
    return None

signature_test = ("modifier", "livre")
print("Canonical pour", signature_test, ":", choisir_canonical(
    catalogue_expose, catalogue_consomme, signature_test
))


Canonical pour ('modifier', 'livre') : valmont_update_book_meta


#### Ce que les trois exemples guidés font des mesures qui les précèdent

Les trois exemples guidés ne sont pas des applications décoratives : chacun reprend une
mesure du corps du notebook et la retourne en décision, en changeant à chaque fois le
niveau de granularité. L'exemple guidé 1 redescend des signatures vers les outils : le
recouvrement de trois signatures, mesuré en section 4, cache quels outils *consommés*
précisément seraient jetables. Sa sortie les nomme — `bookgraph_get_book_info`,
`bookgraph_search_books`, `bookgraph_update_book` — et les quatre outils restants sont
exactement ceux que la section 6 désignait comme apports neufs : les deux mesures se
recoupent outil par outil.

L'exemple guidé 2 isole le sous-ensemble dangereux du recouvrement : parmi les
signatures communes, celles dont le verbe est un verbe d'écriture. Sa sortie n'en
retient qu'une, `(modifier, livre)` — le risque de double-écrit n'est donc pas
hypothétique dans ce jeu de données, il est déjà installé dans le chevauchement mesuré.
Lire deux fois la même fiche coûte une requête ; l'écrire deux fois, par deux chemins
indépendants, coûte une incohérence — c'est toute la différence entre un recouvrement
bénin et un recouvrement qui appelle une politique.

L'exemple guidé 3 tranche la gouvernance du recouvrement : pour la signature
`(modifier, livre)`, il désigne `valmont_update_book_meta`, l'outil interne. La règle,
préférer l'interne, est une politique de souveraineté : ce que le site sait faire seul,
il ne doit pas en dépendre ; la dépendance externe se réserve à ce qu'elle apporte,
mesuré en section 6. Lus ensemble, les trois exemples parcourent le cycle complet d'une
décision d'intégration MCP : *que peut-on retirer* (redondance), *quoi surveiller*
(double-écrit), *qui fait autorité* (canonical).

Ils répondent pourtant chacun à une question isolée, sur un catalogue figé. Les exercices
qui suivent poussent la même démarche dans trois directions qu'ils laissent ouvertes :
**localiser** le recouvrement par cible, **combiner** les réponses en une seule table de
routage, et **réévaluer** la décision quand le serveur externe change de version.

### Exercice 1 — où se concentre le recouvrement ?

L'indice de Jaccard et l'exemple guidé 1 mesurent le recouvrement **globalement**. Une
décision d'intégration a besoin de savoir **sur quelles cibles** il porte : un
recouvrement concentré sur une seule cible appelle une politique locale (une règle pour
les fiches livre), un recouvrement diffus une politique générale.

Écrire `recouvrement_par_cible(expose, consomme)` qui renvoie un dictionnaire
`{cible: {"expose": n1, "consomme": n2, "commun": n3}}` : pour chaque cible présente
dans l'un des deux catalogues, le nombre de signatures exposées, consommées et
communes qui la portent.

- *Indice* : `signatures_catalogue()` donne déjà les deux ensembles de couples `(verbe, cible)`.
- *Étape 1* : réunir les cibles des deux ensembles.
- *Étape 2* : pour chaque cible, compter les signatures qui la portent dans chaque ensemble, puis dans leur intersection.
- *Lecture* : comparer la cible la plus chargée en signatures communes à la conclusion de la section 4.

In [9]:
def recouvrement_par_cible(expose, consomme):
    """Renvoie {cible: {"expose": n1, "consomme": n2, "commun": n3}}.

    n1 / n2 : nombre de signatures (verbe, cible) portant cette cible cote expose / consomme ;
    n3 : nombre de ces signatures presentes des deux cotes.
    """
    # TODO etudiant : parcourir les cibles presentes dans l'un ou l'autre catalogue
    # Indice : signatures_catalogue(expose) et signatures_catalogue(consomme)
    return None


par_cible = recouvrement_par_cible(catalogue_expose, catalogue_consomme)
if par_cible is None:
    print("Exercice a completer")
else:
    for cible in sorted(par_cible):
        print(cible.ljust(14), par_cible[cible])

Exercice a completer


### Exercice 2 — une table de routage qui applique la politique

Les exemples guidés 2 et 3 répondent chacun à une question isolée : *quelles signatures
sont en double-écrit ?* et *quel outil est canonique pour une signature donnée ?*. Un
agent en production a besoin d'une seule table qui applique les deux réponses à la fois,
pour toutes les signatures qu'il peut rencontrer.

Écrire `table_de_routage(expose, consomme)` qui renvoie, pour **chaque signature de
l'union** des deux catalogues, un dictionnaire `{"canonique": nom_outil, "desactives": [noms]}` :

- signature présente d'un seul côté : l'outil de ce côté est canonique, rien n'est désactivé ;
- signature commune avec un verbe de **lecture** : l'interne est canonique, l'externe
  reste disponible (`desactives` vide) — lire deux fois coûte une requête ;
- signature commune avec un verbe d'**écriture** : l'interne est canonique et l'outil
  externe est **désactivé** — écrire deux fois coûte une incohérence.

- *Indice* : réutiliser `detecter_double_ecrit()` et `choisir_canonical()` des exemples guidés.
- *Contrôle* : le nombre d'outils désactivés doit correspondre à la sortie de l'exemple guidé 2.

In [10]:
def table_de_routage(expose, consomme):
    """Renvoie {signature: {"canonique": nom, "desactives": [noms]}} pour l'union des signatures."""
    # TODO etudiant : construire la table a partir des regles de l'enonce
    # Etape 1 : l'union des signatures des deux catalogues
    # Etape 2 : pour chaque signature, choisir l'outil canonique (cf. choisir_canonical)
    # Etape 3 : desactiver l'externe si la signature est en double-ecrit (cf. detecter_double_ecrit)
    return None


routage = table_de_routage(catalogue_expose, catalogue_consomme)
if routage is None:
    print("Exercice a completer")
else:
    for sig in sorted(routage):
        print(sig, "->", routage[sig])

Exercice a completer


### Exercice 3 — auditer une nouvelle version du serveur externe

Le serveur `bookgraph` publie une nouvelle version de son catalogue : deux outils
apparaissent, qui portent sur les **manuscrits** — un objet que la Maison Valmont gère
elle-même. Une signature neuve n'est pas forcément une bonne nouvelle : un outil externe
qui **écrit** sur une cible que Valmont expose déjà ouvre un second chemin d'écriture
vers ses propres données, même si la signature exacte n'existe pas encore côté interne.
L'analyse par signature exacte des exemples guidés ne voit pas ce cas.

Écrire `classer_nouveautes(expose, consomme_avant, consomme_apres)` qui classe chaque
signature **apparue** entre les deux versions dans l'une de quatre catégories :

| Catégorie | Condition |
|---|---|
| `"lecture redondante"` | signature déjà exposée, verbe de lecture |
| `"double-ecrit"` | signature déjà exposée, verbe d'écriture |
| `"ecriture sur cible souveraine"` | signature non exposée, verbe d'écriture, cible déjà exposée par Valmont |
| `"capacite neuve"` | tous les autres cas |

- *Indice* : la cible d'une signature est son second élément ; `VERBES_ECRITURE` (exemple guidé 2) sert de référence pour les verbes d'écriture.
- *Étape 1* : calculer les signatures apparues entre les deux versions.
- *Étape 2* : appliquer le tableau ligne par ligne, dans l'ordre.

In [11]:
# Nouvelle version du catalogue consomme : deux outils apparaissent
catalogue_consomme_v2 = dict(catalogue_consomme)
catalogue_consomme_v2.update({
    "bookgraph_get_manuscript":    {"verbe": "lire",     "cible": "manuscrit"},
    "bookgraph_update_manuscript": {"verbe": "modifier", "cible": "manuscrit"},
})


def classer_nouveautes(expose, consomme_avant, consomme_apres):
    """Renvoie {signature_apparue: categorie} -- cf. le tableau de l'enonce."""
    # TODO etudiant : classer chaque signature apparue entre les deux versions
    # Indice : sig[1] est la cible ; VERBES_ECRITURE distingue les ecritures
    return None


print("Outils consommes : " + str(len(catalogue_consomme)) + " -> " + str(len(catalogue_consomme_v2)))
classement = classer_nouveautes(catalogue_expose, catalogue_consomme, catalogue_consomme_v2)
if classement is None:
    print("Exercice a completer")
else:
    for sig in sorted(classement):
        print(sig, "->", classement[sig])

Outils consommes : 7 -> 9
Exercice a completer


## Conclusion

Ce carnet a remplace un exercice en memoire, puis une mesure contre un serveur ecrit par le carnet lui-meme, par une mesure contre **deux serveurs reels et externes**. Ce qu'il a fallu pour cela, et ce que la mesure a appris :

1. **Les deux sens sont reels, et aucun n'est construit par la mesure.** Le cote expose est le catalogue effectif d'AI Engine — 42 outils, tous generiques, aucun outil metier — lu sur la route miroir **et** consomme par le protocole MCP lui-meme (session JSON-RPC, `tools/list`, `tools/call`). Le second serveur est le `filesystem` officiel du projet MCP, joint par stdio, qui tient un rayon de fiches de lecture. La batterie oppose donc deux systemes independants, comme dans une vraie orchestration.
2. **La forme ne classe pas le risque.** Jaccard place toutes les paires de meme verbe au meme coefficient : il ne distingue pas une lecture d'une ecriture, ni une fiche d'un article, et sa signature ne porte aucune trace de l'origine. Il reste utile comme premier tri, il ne peut pas etre l'aboutissement.
3. **Le comportement se mesure, avec ses intervalles.** Face aux 55 outils fusionnes, chaque tache repetee donne un taux avec son intervalle de Wilson ; et la meme batterie passee sur deux moteurs de la meme famille separe ce qui vient du **modele** de ce qui vient du **protocole** — si les deux moteurs traversent la frontiere de la meme facon, ce n'est pas un defaut de capacite. Les chiffres de cette execution sont dans la section 5 : ils dependent du moteur et de la formulation, et c'est justement l'interet — la mesure se rejoue, un resultat ne se transporte pas d'une execution a l'autre.
4. **L'echec n'est pas toujours la double ecriture.** Une traversee est un evenement parmi d'autres : l'agent peut aussi se tromper d'outil dans le bon systeme, ou ne rien appeler du tout. Ces trois verdicts n'appellent pas la meme correction — une regle de propriete par systeme pour le premier, un catalogue plus lisible pour le deuxieme, une consigne plus precise pour le troisieme.
5. **La consigne de routage rend la regle explicite, elle ne garantit pas qu'elle soit suivie.** L'exemple 3 compare les taux de traversee avec et sans politique, intervalles compris. Un agent appele a ecrire dans deux systemes demande une regle de propriete par systeme — ecrite, comprise, et re-mesuree a chaque changement de modele.

### Provenance et limites

- **Cote expose/consomme site** : plugin `ai-engine` 3.7.0 de l'instance jetable — catalogue lu sur `mwai/v1/mcp/functions`, session MCP consommee sur `mcp/v1/http` (JSON-RPC, protocole negocie 2025-06-18, mot de passe d'application).
- **Cote rayon** : serveur `filesystem` officiel (`@modelcontextprotocol/server-filesystem`), lance par `npx -y`, joint par stdio.
- **Moteurs** : deux vLLM locaux de la meme famille de modeles, tailles differentes (noms et comptes en section 5), appel d'outils natif active.
- **Limites** : la batterie mesure des **choix** (l'outil que le modele emet), pas des executions — aucun `tools/call` n'est declenche pendant la batterie ; seuls les exemples ecrivent reellement. Les intervalles de Wilson supposent des essais independants ; l'instance jetable se recree a l'identique mais les moteurs, eux, evoluent.

La frontiere gratuite mesuree ailleurs dans la serie reste valable : l'instance gratuite ne fournit aucune route pour **declarer** un serveur MCP client. Le cote consomme de ce carnet consomme donc le serveur que le plugin expose lui-meme, plus un serveur externe que nous lancons — ce qui est exactement la situation reelle d'une plateforme qui consomme des outils qu'elle n'heberge pas.

**Pour aller plus loin.**
- [`auditer-un-serveur-mcp.ipynb`](auditer-un-serveur-mcp.ipynb) — l'autre
  moitié : classifier les outils d'un **seul** catalogue en CRUD générique vs
  verbe métier (ce notebook compare **deux** catalogues).
- [`livresagites-parcours.md`](../../04-Cas-Usage-livresagites/livresagites-parcours.md) Parcours 0 — la
  confusion fréquente entre les deux sens du protocole, dont ce notebook est
  l'illustration exécutable.